In [19]:
!pip install duckdb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 23.4 MB/s eta 0:00:00:00:0100:01

[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [32]:





valid_folder = {'ingest_date_20260701_102928': datetime.datetime(2026, 7, 1, 10, 29, 28),
 'ingest_date_20260701_103044': datetime.datetime(2026, 7, 1, 10, 30, 44)}




    #break
 #dj


data/full_load/ingest_date_20260701_102928/part_001.parquet
1
data/full_load/ingest_date_20260701_103044/part_001.parquet
1
data/full_load/ingest_date_20260701_103044/part_002.parquet
1
data/full_load/ingest_date_20260701_103044/part_003.parquet
1
data/full_load/ingest_date_20260701_103044/part_004.parquet
1
data/full_load/ingest_date_20260701_103044/part_005.parquet
1


In [33]:
con = duckdb.connect("mydb.duckdb")

df = con.execute("select * from staging_bronze").df()
df.shape

(115586, 45)

In [1]:
import boto3
import duckdb
import tempfile
import datetime
import pandas as pd

s3 = boto3.client("s3")
con = duckdb.connect()

bucket = "nyi-data-analytics"

valid_folder = {
    'ingest_date_20260701_102928': datetime.datetime(2026, 7, 1, 10, 29, 28),
    'ingest_date_20260701_103044': datetime.datetime(2026, 7, 1, 10, 30, 44)
}

dfs = []

for folder_name, ts in valid_folder.items():

    prefix = f"data/full_load/{folder_name}/"
    resp = s3.list_objects_v2(Bucket=bucket, Prefix=prefix)

    for obj in resp.get("Contents", []):
        key = obj["Key"]

        if not key.endswith(".parquet"):
            continue

        file_obj = s3.get_object(Bucket=bucket, Key=key)

        with tempfile.NamedTemporaryFile(suffix=".parquet") as f:
            f.write(file_obj["Body"].read())
            f.flush()

            df = con.execute("""
                SELECT *,
                       ? AS ingest_timestamp
                FROM read_parquet(?, union_by_name=true)
            """, [ts, f.name]).df()

            dfs.append(df)

# Final DataFrame
final_df = pd.concat(dfs, ignore_index=True)

In [2]:
final_df.head()

,unique_key,created_date,closed_date,agency,agency_name,complaint_type,descriptor,descriptor_2,incident_zip,incident_address,...,landmark,due_date,vehicle_type,taxi_pick_up_location,bridge_highway_name,bridge_highway_direction,bridge_highway_segment,road_ramp,taxi_company_borough,ingest_timestamp
0,68537299,2026-04-02T11:28:00.000,2026-04-02T11:39:00.000,DEP,Department of Environmental Protection,Water System,Excessive Water In Basement (WEFB),WEFB,11238,302 LAFAYETTE AVENUE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-07-01 10:29:28
1,68537302,2026-04-02T12:12:00.000,2026-04-02T13:20:00.000,DEP,Department of Environmental Protection,Water Conservation,"Wasting Faucets,Sinks,Flushometer,Urinal,Etc. ...",CWR,11435,142-15 110 AVENUE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-07-01 10:29:28
2,68537303,2026-04-02T11:34:00.000,2026-04-02T15:06:00.000,DEP,Department of Environmental Protection,Water Conservation,"Wasting Faucets,Sinks,Flushometer,Urinal,Etc. ...",CWR,10023,2 COLUMBUS AVENUE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-07-01 10:29:28
3,68537304,2026-04-02T13:55:00.000,2026-04-08T00:00:00.000,DEP,Department of Environmental Protection,Noise,Noise: air condition/ventilation equipment (NV1),NV1,10019,1000 10 AVENUE,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-07-01 10:29:28
4,68537311,2026-04-02T21:39:00.000,2026-04-06T13:00:00.000,DEP,Department of Environmental Protection,Noise,"Noise, Barking Dog (NR5)",NR5,11201,73 COLUMBIA STREET,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-07-01 10:29:28


In [8]:
#final_df.to_csv("data_analyse.csv")
final_df.dtypes.astype(str).to_dict()

{'unique_key': 'str',
 'created_date': 'str',
 'closed_date': 'str',
 'agency': 'str',
 'agency_name': 'str',
 'complaint_type': 'str',
 'descriptor': 'str',
 'descriptor_2': 'str',
 'incident_zip': 'str',
 'incident_address': 'str',
 'street_name': 'str',
 'cross_street_1': 'str',
 'cross_street_2': 'str',
 'address_type': 'str',
 'city': 'str',
 'status': 'str',
 'resolution_description': 'str',
 'resolution_action_updated_date': 'str',
 'community_board': 'str',
 'council_district': 'str',
 'police_precinct': 'str',
 'bbl': 'str',
 'borough': 'str',
 'x_coordinate_state_plane': 'str',
 'y_coordinate_state_plane': 'str',
 'open_data_channel_type': 'str',
 'park_facility_name': 'str',
 'park_borough': 'str',
 'latitude': 'str',
 'longitude': 'str',
 'location': 'object',
 'facility_type': 'str',
 'intersection_street_1': 'str',
 'intersection_street_2': 'str',
 'location_type': 'str',
 'landmark': 'str',
 'due_date': 'str',
 'vehicle_type': 'str',
 'taxi_pick_up_location': 'str',
 'br

In [79]:
print(columns)

[('unique_key', 'VARCHAR', 'YES', None, None, None), ('created_date', 'VARCHAR', 'YES', None, None, None), ('closed_date', 'VARCHAR', 'YES', None, None, None), ('agency', 'VARCHAR', 'YES', None, None, None), ('agency_name', 'VARCHAR', 'YES', None, None, None), ('complaint_type', 'VARCHAR', 'YES', None, None, None), ('descriptor', 'VARCHAR', 'YES', None, None, None), ('descriptor_2', 'VARCHAR', 'YES', None, None, None), ('incident_zip', 'VARCHAR', 'YES', None, None, None), ('incident_address', 'VARCHAR', 'YES', None, None, None), ('street_name', 'VARCHAR', 'YES', None, None, None), ('cross_street_1', 'VARCHAR', 'YES', None, None, None), ('cross_street_2', 'VARCHAR', 'YES', None, None, None), ('address_type', 'VARCHAR', 'YES', None, None, None), ('city', 'VARCHAR', 'YES', None, None, None), ('status', 'VARCHAR', 'YES', None, None, None), ('resolution_description', 'VARCHAR', 'YES', None, None, None), ('resolution_action_updated_date', 'VARCHAR', 'YES', None, None, None), ('community_boar

In [6]:
state["last_created_date"] = "2026-06-28"
state["last_unique_key"] = 234567899
state["last_ingest_time"] = datetime.now().isoformat()

with open("current_state.json", "w") as f:
    json.dump(state, f, indent=4)

print(state)

def extract_data():
    base_url = "https://data.cityofnewyork.us/api/odata/v4/erm2-nwe9"
    date_cutoff = datetime.now() - timedelta(days=90)
    date_str = date_cutoff.strftime("%Y-%m-%d")
    url = f"{base_url}?$filter=created_date ge {date_str}&$format=json"
    

In [3]:
date_cutoff = datetime.now() - timedelta(days=90)
date_str = date_cutoff.strftime("%Y-%m-%dT00:00:00.000")

In [ ]:
base_url = "https://data.cityofnewyork.us/api/odata/v4/erm2-nwe9"
date_cutoff = datetime.now() - timedelta(days=90)
date_str = date_cutoff.strftime("%Y-%m-%dT00:00:00.000")
url = (
    "https://data.cityofnewyork.us/resource/erm2-nwe9.json"
    f"?$select=count(*)&$where=created_date >= '{date_str}'"
)


all_data = []

while url:
    response = requests.get(url)
    data = response.json()

    all_data.extend(data["value"])
    url = data.get("@odata.nextLink")

df = pd.DataFrame(all_data)

print(df.len())
df.head()

In [ ]:
all_data = []

while url:
    response = requests.get(url)
    data = response.json()

    all_data.extend(data["value"])
    url = data.get("@odata.nextLink")

df = pd.DataFrame(all_data)

df.head() 


In [ ]:
import pyarrow 
import boto3
import s3fs
import awscli



In [ ]:
s3 = boto3.client('s3')
response = s3.list_buckets()
for i in response['Buckets']:
    print(i['Name'])

nyi-data-analytics


In [3]:
s3.upload_file(
    'chcek.csv', #file in our system
    'nyi-data-analytics' , #bucket name
    'first_thousand'
)

In [3]:
import pyarrow 
import boto3
import s3fs
import awscli
from io import BytesIO
from datetime import datetime , timedelta
from zoneinfo import ZoneInfo


s3 = boto3.client('s3')
response = s3.list_buckets()
#print(Bucket Name)
for i in response['Buckets']:
    print(i['Name'])



nyi-data-analytics


In [7]:
def generate_folder_name() :
    timestamp = datetime.now()
    ts= timestamp.strftime("%Y%m%d_%H%M%S")
    return f"ingest_date_{ts}"

folder_name = generate_folder_name()
print(folder_name)

ingest_date_20260701_092624


In [19]:
def upload_df_to_s3(df, bucket_name, base_prefix, folder_name, file_name):
    
    buffer = BytesIO()

    #dataframe of raw_data
    #converts raw data to parquet and stores it in ram instead of Disk
    df.to_parquet(
        buffer,
        engine="pyarrow",
        index=False
    )
    
    #to bring cursor on start
    buffer.seek(0)

    #to create file path = object key
    key = f"{base_prefix}/{folder_name}/{file_name}.parquet"

    
    s3.upload_fileobj(
        buffer,
        bucket_name,
        key
    )

    return f"s3://{bucket_name}/{key}"

def generate_folder_name() :
    timestamp = datetime.now(ZoneInfo("Asia/Kolkata"))
    ts= timestamp.strftime("%Y%m%d_%H%M%S")
    return f"ingest_date_{ts}"

folder_name = generate_folder_name()
file_load = 1
s3 = boto3.client('s3')

df = pd.read_csv("chcek.csv")
upload_df_to_s3(df, 
                bucket_name = 'nyi-data-analytics',
                base_prefix = 'data/full_load',
                folder_name = folder_name,
                file_name = f"part_{file_load:03d}")
                

's3://nyi-data-analytics/data/full_load/ingest_date_20260701_095542/part_001.parquet'

In [10]:
import pandas as pd
df3=pd.read_csv("chcek.csv")
df3.shape

(1000, 45)

In [17]:
buffer = BytesIO()
df.to_parquet(
        buffer,
        engine="pyarrow",
        index=False
    )

#to bring cursor on start
buffer.seek(0)

    key = f"{base_prefix}/{folder_name}/{file_name}.parquet"

    s3_client.upload_fileobj(
        buffer,
        bucket_name,
        key
    )

    return f"s3://{bucket_name}/{key}"


,Unnamed: 0,unique_key,created_date,closed_date,agency,agency_name,complaint_type,descriptor,descriptor_2,location_type,...,vehicle_type,taxi_company_borough,taxi_pick_up_location,bridge_highway_name,bridge_highway_direction,road_ramp,bridge_highway_segment,latitude,longitude,location
0,0,69396322,2026-06-18T02:06:17.000,NaN,NYPD,New York City Police Department,Illegal Parking,Double Parked Blocking Traffic,NaN,Street/Sidewalk,...,Car,NaN,NaN,NaN,NaN,NaN,NaN,40.644248,-73.880099,POINT (-73.88009914733 40.644247849899)
1,1,69390855,2026-06-18T02:05:50.000,NaN,DSNY,Department of Sanitation,Dirty Condition,Trash,Littering,Sidewalk,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.800151,-73.953215,POINT (-73.95321534259 40.800150933593)
2,2,69396370,2026-06-18T02:05:49.000,NaN,NYPD,New York City Police Department,Noise - Residential,Loud Talking,NaN,Residential Building/House,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.646580,-73.950087,POINT (-73.950087024662 40.646580439376)
3,3,69397701,2026-06-18T02:05:36.000,NaN,NYPD,New York City Police Department,Noise - Street/Sidewalk,Loud Music/Party,NaN,Street/Sidewalk,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.693884,-74.000844,POINT (-74.000843832356 40.693883764345)
4,4,69400344,2026-06-18T02:05:24.000,NaN,NYPD,New York City Police Department,Drug Activity,Use Indoor,NaN,Other,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.806681,-73.927130,POINT (-73.92713042937 40.806680823068)


In [21]:
df_3 = pd.read_parquet("part_001.parquet")
df_3.head()

,Unnamed: 0,unique_key,created_date,closed_date,agency,agency_name,complaint_type,descriptor,descriptor_2,location_type,...,vehicle_type,taxi_company_borough,taxi_pick_up_location,bridge_highway_name,bridge_highway_direction,road_ramp,bridge_highway_segment,latitude,longitude,location
0,0,69396322,2026-06-18T02:06:17.000,NaN,NYPD,New York City Police Department,Illegal Parking,Double Parked Blocking Traffic,NaN,Street/Sidewalk,...,Car,NaN,NaN,NaN,NaN,NaN,NaN,40.644248,-73.880099,POINT (-73.88009914733 40.644247849899)
1,1,69390855,2026-06-18T02:05:50.000,NaN,DSNY,Department of Sanitation,Dirty Condition,Trash,Littering,Sidewalk,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.800151,-73.953215,POINT (-73.95321534259 40.800150933593)
2,2,69396370,2026-06-18T02:05:49.000,NaN,NYPD,New York City Police Department,Noise - Residential,Loud Talking,NaN,Residential Building/House,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.646580,-73.950087,POINT (-73.950087024662 40.646580439376)
3,3,69397701,2026-06-18T02:05:36.000,NaN,NYPD,New York City Police Department,Noise - Street/Sidewalk,Loud Music/Party,NaN,Street/Sidewalk,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.693884,-74.000844,POINT (-74.000843832356 40.693883764345)
4,4,69400344,2026-06-18T02:05:24.000,NaN,NYPD,New York City Police Department,Drug Activity,Use Indoor,NaN,Other,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.806681,-73.927130,POINT (-73.92713042937 40.806680823068)


In [11]:
import pandas as pd
df= pd.read_parquet("part_017.parquet")
df.columns

Index(['unique_key', 'created_date', 'closed_date', 'agency', 'agency_name',
       'complaint_type', 'descriptor', 'location_type', 'incident_zip',
       'incident_address', 'street_name', 'cross_street_1', 'cross_street_2',
       'intersection_street_1', 'intersection_street_2', 'address_type',
       'city', 'landmark', 'status', 'resolution_description',
       'resolution_action_updated_date', 'community_board', 'council_district',
       'police_precinct', 'borough', 'x_coordinate_state_plane',
       'y_coordinate_state_plane', 'open_data_channel_type',
       'park_facility_name', 'park_borough', 'latitude', 'longitude',
       'location', 'bbl', 'vehicle_type', 'descriptor_2', 'facility_type',
       'taxi_pick_up_location', 'bridge_highway_name',
       'bridge_highway_direction', 'bridge_highway_segment',
       'taxi_company_borough'],
      dtype='str')

In [12]:
df2 = pd.read_csv("data_analyse.csv")
df2.columns

/tmp/ipykernel_41/2325832937.py:1: DtypeWarning: Columns (0: road_ramp, 1: taxi_company_borough) have mixed types. Specify dtype option on import or set low_memory=False.
  df2 = pd.read_csv("data_analyse.csv")


Index(['Unnamed: 0', 'unique_key', 'created_date', 'closed_date', 'agency',
       'agency_name', 'complaint_type', 'descriptor', 'descriptor_2',
       'incident_zip', 'incident_address', 'street_name', 'cross_street_1',
       'cross_street_2', 'address_type', 'city', 'status',
       'resolution_description', 'resolution_action_updated_date',
       'community_board', 'council_district', 'police_precinct', 'bbl',
       'borough', 'x_coordinate_state_plane', 'y_coordinate_state_plane',
       'open_data_channel_type', 'park_facility_name', 'park_borough',
       'latitude', 'longitude', 'location', 'facility_type',
       'intersection_street_1', 'intersection_street_2', 'location_type',
       'landmark', 'due_date', 'vehicle_type', 'taxi_pick_up_location',
       'bridge_highway_name', 'bridge_highway_direction',
       'bridge_highway_segment', 'road_ramp', 'taxi_company_borough',
       'ingest_timestamp'],
      dtype='str')